<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 55%,#00A86A 100%);padding:30px 34px;border-radius:14px;border-bottom:7px solid #F5C242;color:#FFFFFF">
<div style="color:#F5C242;font-weight:700;letter-spacing:3px;font-size:12px">STG17 TECHNICAL WORKSHOP · DAY 2 · 14:00–14:45 · LABORATORY</div>
<h1 style="color:#FFFFFF;margin:10px 0 6px 0;font-size:38px">From Good to Great</h1>
<h3 style="color:#E6F6EE;margin:0 0 14px 0;font-weight:400"><i>Prompt optimisation for official statistics: a hands-on notebook</i></h3>
<div style="color:#E6F6EE;font-size:13px">Emerging Issues, Emerging Practice · Innovating the Data Value Chain<br>African Development Bank · African Union (STATAFRIC) · National Institute of Statistics of Rwanda</div>
</div>

## Why this notebook?

This morning you learnt how to **write** a good prompt. This notebook answers the next question, the one every national statistical office (NSO) faces before putting an LLM into production:

> **How do you prove a prompt works, improve it in a controlled way, and keep it affordable?**

We work on a concrete task most offices know: **coding free-text occupations to ISCO-08** in a labour force survey, where enumerators type answers in French or English.

<div style="background:#E8F5EF;border-left:5px solid #00A86A;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>🎯 By the end of this notebook you will be able to:</b><br>
1. build a <b>frozen, representative evaluation set</b>;<br>
2. <b>measure</b> prompt quality with the right metric, and keep an iteration log;<br>
3. <b>control variance</b> (temperature, majority vote);<br>
4. estimate and reduce <b>token cost</b> (caching, batching, shorter context);<br>
5. choose between <b>prompting, RAG and fine-tuning</b> according to the failure you observe.
</div>

### Contents

| Section | Content | Key concept |
|---|---|---|
| **0** | Setup and configuration | Colab, Kaggle or local; live or simulation mode |
| **1** | From good to great | One example proves nothing |
| **2** | The evaluation set | The frozen test that turns opinions into numbers |
| **3** | Measuring quality | Metrics, optimisation loop, log, data leakage |
| **4** | Controlling variance | Temperature, consistency test, majority vote |
| **5** | Context, tokens and cost | Tokens, cost, batching, caching, “lost in the middle” |
| **6** | Prompt, RAG or fine-tune? | Mini-RAG, LLM judge, decision path |
| **7** | 🧪 Your turn | Optimise the prompt yourself |
| **8** | Wrap-up and export | Checklist, files for the 14:45 benchmark |

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>⏱️ How to use this notebook:</b> run the cells <b>in order</b> (<i>Runtime → Run all</i> also works). Without an API key it runs in <b>simulation mode</b>: a teaching simulator imitates an LLM and reacts to the changes you make to the prompt. With a Groq key, everything runs on a real model.
</div>

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">00</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Setup and configuration</span><br>
<i style="color:#E6F6EE">One notebook for Colab, Kaggle and a local install.</i>
</div>

### 0.1 Install the libraries

The next cell installs only what is missing. It does nothing if everything is already present, which is usually the case on Colab and Kaggle.

In [ ]:
# ⚙️ Install missing dependencies
import sys, os, subprocess, importlib

def _install(package):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

for module, package in [("openai", "openai>=1.40"), ("tiktoken", "tiktoken"), ("pandas", "pandas"),
                        ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installing {package}…")
        _install(package)

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ENVIRONMENT = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local / other")
print(f"✅ Dependencies ready · Environment detected: {ENVIRONMENT} · Python {sys.version.split()[0]}")

### 0.2 Choose the model provider

| Option | When to use it | Key required |
|---|---|---|
| `"groq"` | Recommended for the workshop (same engine as the 14:45 session) | `GROQ_API_KEY` |
| `"openai"` | Any OpenAI-compatible provider | `OPENAI_API_KEY` |
| `"ollama"` | An open-weight model **hosted on your machine**: data never leaves it | none |
| `"simulation"` | No key, no connection: teaching simulator | none |
| `"auto"` | Groq if a key is found, otherwise simulation | — |

**Where to put the key**
- **Colab**: the 🔑 *Secrets* icon on the left → add `GROQ_API_KEY` → grant notebook access.
- **Kaggle**: *Add-ons → Secrets* → add `GROQ_API_KEY`.
- **Local**: environment variable `export GROQ_API_KEY=...` before starting Jupyter.

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>🔒 Confidentiality:</b> never send identifiable microdata to an external API without checking the legal basis and the data terms first. All data in this notebook is <b>fictional</b>. Never paste an API key in clear text into a cell you share.
</div>

In [ ]:
# 🔧 CONFIGURATION: the only cell you need to edit
PROVIDER = "auto"          # "auto" | "groq" | "openai" | "ollama" | "simulation"

MODELS = {
    "groq":   "openai/gpt-oss-20b",   # Groq production model (check the list in cell 0.3)
    "openai": "TO_BE_FILLED_IN",      # exact model name at your provider
    "ollama": "llama3.2",             # model pulled locally with `ollama pull`
}
OPENAI_COMPATIBLE_URL = None      # e.g. the URL of another OpenAI-compatible provider

# ILLUSTRATIVE prices in US dollars per million tokens: replace with your provider's real tariff
PRICE_IN_PER_MILLION = 0.50
PRICE_OUT_PER_MILLION = 1.50

ASK_KEY_IF_MISSING = False   # True: masked prompt for the key if none is found
PAUSE_BETWEEN_CALLS_S = 0.3  # small pause to stay within free-tier rate limits

In [ ]:
# 🔑 Safe key lookup (environment variables, Colab or Kaggle secrets)
import getpass

def read_key(name):
    value = os.environ.get(name)
    if value:
        return value
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass
    if ASK_KEY_IF_MISSING:
        return getpass.getpass(f"{name} (hidden input): ") or None
    return None

GROQ_KEY = read_key("GROQ_API_KEY") if PROVIDER in ("auto", "groq") else None
OPENAI_KEY = read_key("OPENAI_API_KEY") if PROVIDER == "openai" else None

if PROVIDER == "auto":
    ACTIVE_PROVIDER = "groq" if GROQ_KEY else "simulation"
else:
    ACTIVE_PROVIDER = PROVIDER
if ACTIVE_PROVIDER == "groq" and not GROQ_KEY:
    print("⚠️ No Groq key found: switching to simulation mode.")
    ACTIVE_PROVIDER = "simulation"

ACTIVE_MODEL = MODELS.get(ACTIVE_PROVIDER, "teaching-simulator-v1")
SIMULATION_MODE = ACTIVE_PROVIDER == "simulation"
print(f"Provider: {ACTIVE_PROVIDER} · Model: {ACTIVE_MODEL}")
if SIMULATION_MODE:
    print("ℹ️ Simulation mode: answers come from a teaching simulator, not from a real LLM.")

### 0.3 Toolbox: styling, token counting, LLM client and simulator

The next cells define the tools used throughout the notebook. **You do not need to read them in detail**: just run them. All you need to remember is what the LLM client does:

- it sends messages to the chosen provider and **measures** input and output tokens, latency and estimated cost;
- it **caches** answers obtained at temperature 0 (section 5 explains why);
- it **logs every call**, for auditing and cost accounting;
- it handles **rate limits** with spaced retries.

In [ ]:
# 🎨 Styling (AfDB-inspired palette) and display helpers
import json, re, time, hashlib, random, math
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

GREEN, DEEP_GREEN, FOREST, GOLD, OCHRE = "#00A86A", "#00704A", "#00553A", "#F5C242", "#D49A00"
TEAL, TERRA, BRICK, INK, SLATE, SAGE = "#0E7C86", "#C4621D", "#B83B2E", "#231F20", "#5E6964", "#D5DED9"
RAMP = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10.5, "axes.edgecolor": SAGE, "axes.labelcolor": SLATE,
    "xtick.color": SLATE, "ytick.color": SLATE, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlecolor": INK,
})
pd.set_option("display.max_colwidth", 80)

def callout(text, kind="note"):
    styles = {"note": ("#E8F5EF", GREEN, "💡"), "warning": ("#FBF1D9", OCHRE, "⚠️"),
              "remember": ("#F4F7F5", DEEP_GREEN, "📌"), "risk": ("#F6E3E0", BRICK, "⛔")}
    fill, border, icon = styles[kind]
    display(HTML(f'<div style="background:{fill};border-left:5px solid {border};padding:10px 14px;'
                 f'border-radius:6px;margin:8px 0;color:{INK}">{icon} {text}</div>'))

In [ ]:
# 🔢 Token counting
try:
    import tiktoken
    _encoder = tiktoken.get_encoding("o200k_base")
    def count_tokens(text):
        return len(_encoder.encode(text))
    TOKEN_METHOD = "tiktoken o200k_base (close to the gpt-oss tokeniser)"
except Exception:
    def count_tokens(text):
        return max(1, round(len(text) / 4))
    TOKEN_METHOD = "approximation: 4 characters ≈ 1 token (tiktoken unavailable)"

def usd_cost(input_tokens, output_tokens):
    return input_tokens / 1e6 * PRICE_IN_PER_MILLION + output_tokens / 1e6 * PRICE_OUT_PER_MILLION

print("Counting method:", TOKEN_METHOD)
print("Example:", count_tokens("The consumer price index rose by 2.4 percent in June."), "tokens")

In [ ]:
# 🤖 Teaching simulator: imitates an LLM and reacts to the content of the prompt.
# It "understands" nothing: it applies rules that reproduce behaviours typical of LLMs
# (invented codes without a code list, over-interpretation of vague cases, confusion
# between neighbouring categories, sensitivity to temperature, and so on).

class LLMSimulator:
    KEYWORDS = [
        ("2330", ["lycée", "secondary", "collège"]),
        ("2341", ["primary", "primaire", "instituteur", "institutri"]),
        ("2359", ["cours particuliers", "tutor"]),
        ("2211", ["docteur", "médecin", "doctor"]),
        ("2221", ["infirmi", "nurse"]),
        ("2512", ["software", "logiciel", "développeur", "developer"]),
        ("2411", ["comptable", "accountant"]),
        ("4132", ["saisie", "data entry"]),
        ("4110", ["administratif", "office clerk"]),
        ("5120", ["cook", "cuisinier", "cuisinière"]),
        ("5141", ["coiff", "hairdress"]),
        ("5311", ["nounou", "garde les enfants", "nanny", "child care"]),
        ("5414", ["gardien", "security", "vigile"]),
        ("5223", ["supermarket", "supermarché", "shop assistant"]),
        ("STREET_FOOD", ["beignet", "arachide", "cacahu", "grillé", "roasted"]),
        ("STREET", ["recharge", "airtime", "phone credit", "dans la rue", "on the street"]),
        ("5211", ["marché", "market"]),
        ("6121", ["chèvre", "vache", "bétail", "livestock", "cattle"]),
        ("6111", ["cultive", "maïs", "manioc", "farm", "champ"]),
        ("7112", ["maçon", "bricklayer", "mason"]),
        ("7115", ["menuisier", "charpentier", "carpenter"]),
        ("7231", ["mécanicien", "mechanic"]),
        ("7411", ["électricien", "electrician"]),
        ("7512", ["boulanger", "baker"]),
        ("7531", ["couturi", "tailleur", "tailor"]),
        ("8321", ["moto", "motorcycle"]),
        ("8322", ["taxi", "chauffeur", "driver"]),
        ("9112", ["cleaner", "nettoy", "entretien"]),
    ]
    VAGUE = {"ingénieur": "2142", "works": "9629", "aide ses parents": "6111",
             "business": "1420", "fonctionnaire": "4110", "travailleur": "9629"}
    MULTI_MARKERS = ["also", "aussi", "weekend", "week-end", "le soir"]

    @staticmethod
    def _h(text):
        return int(hashlib.sha256(text.encode()).hexdigest(), 16) % 100

    @staticmethod
    def _neighbour(code):
        neighbours = {"5211": "5221", "8321": "8322", "8322": "8321", "6111": "6121", "6121": "6111",
                      "2341": "2330", "2330": "2341", "7115": "7522", "7112": "7114", "5120": "5131",
                      "2221": "3221", "2211": "2240", "4132": "4131", "5414": "5419", "9112": "9111"}
        if code in neighbours:
            return neighbours[code]
        if code and code.isdigit():
            return code[:3] + str((int(code[3]) + 1) % 10)
        return "9629"

    def _keyword(self, t):
        for code, words in self.KEYWORDS:
            if any(w in t for w in words):
                return code
        return None

    def _features(self, prompt):
        p = prompt.lower()
        examples = {m.group(1).strip().lower(): m.group(2) for m in re.finditer(r'-\s*"([^"]+)"\s*→\s*(\w+)', prompt)}
        return {
            "list": "9520" in prompt and "8321" in prompt,
            "json": "json" in p,
            "schema": '"isco_code"' in prompt,
            "main_job": "main job" in p or "emploi principal" in p,
            "uncodable": "UNCODABLE" in prompt and ("too vague" in p or "unclear" in p),
            "supermarket": bool(re.search(r"supermarket|supermarch", re.sub(r'-\s*"[^"]+"\s*→\s*\w+', "", p))),
            "examples": examples,
        }

    def _code_one(self, text, ft, temperature, rng, without_examples=False):
        t = text.strip().lower()
        examples = {} if without_examples else ft["examples"]
        example_codes = set(examples.values())
        if t in examples:                                    # the case is in the prompt: "leakage"
            return examples[t], False
        if t in self.VAGUE:
            return ("UNCODABLE" if ft["uncodable"] else self.VAGUE[t]), False
        multi = "," in t and any(m in t for m in self.MULTI_MARKERS)
        if multi:
            main, secondary = t.split(",", 1)
            c1, c2 = self._keyword(main), self._keyword(secondary)
            raw = c1 if (ft["main_job"] or c2 is None) else c2
        else:
            raw = self._keyword(t)
        code = raw
        if raw == "STREET_FOOD":
            code = "5212" if "5212" in example_codes else "5211"
        elif raw == "STREET":
            code = "9520" if "9520" in example_codes else "5211"
        elif raw == "5223" and not ft["supermarket"]:
            code = "5211"
        if code is None:
            code = "9629"
        h = self._h(t)
        if not ft["list"] and h < 45:
            code = code[:3] + "0"                            # invented code or wrong level of detail
        elif h < 6:
            code = self._neighbour(code)                     # genuinely hard case
        if temperature > 0:
            p = temperature * (0.35 if (multi or raw in ("STREET", "STREET_FOOD", "5223")) else 0.12)
            if rng.random() < p:
                code = rng.choice([self._neighbour(code), "5211", "9629"])
        return code, multi

    def _coding(self, messages, prompt, temperature, rng):
        system = messages[0]["content"] if messages[0]["role"] == "system" else ""
        ft = self._features(system)   # the simulator only "reads" the instructions, not the case
        user = messages[-1]["content"]
        batch = re.findall(r"^\[(\w+)\]\s*(.+)$", user, flags=re.M)
        if batch:
            results = []
            for i, (case_id, text) in enumerate(batch):
                code, _ = self._code_one(text, ft, temperature, rng, without_examples=(i >= 8))
                results.append({"id": case_id, "isco_code": code})
            return json.dumps({"results": results}, ensure_ascii=False)
        m = re.search(r"Description\s*:\s*(.+)", user)
        text = m.group(1) if m else re.split(r":\s*", user)[-1]
        code, multi = self._code_one(text, ft, temperature, rng)
        confidence = "low" if code == "UNCODABLE" else ("medium" if multi else "high")
        if ft["json"] and ft["schema"]:
            return json.dumps({"isco_code": code, "confidence": confidence}, ensure_ascii=False)
        if ft["json"]:
            return json.dumps({"code": code}, ensure_ascii=False)
        if not ft["list"]:
            return (f"This occupation probably corresponds to ISCO code {code}. "
                    "You should nevertheless check the official classification, as several "
                    "groups may fit depending on the context of the job.")
        return f"Based on the list provided, the most appropriate ISCO-08 code is {code}."

    def _needle(self, prompt):
        target = re.search(r"March stands at ([\d.,]+) %", prompt)
        decoy = re.search(r"February stands at ([\d.,]+) %", prompt)
        if not target:
            return "I cannot find this information in the bulletin."
        position = target.start() / max(1, len(prompt))
        long_context = count_tokens(prompt) > 1500
        if long_context and 0.25 < position < 0.75 and self._h(target.group(1)) % 3 != 0 and decoy:
            return f"{decoy.group(1)} %"
        return f"{target.group(1)} %"

    def _judge(self, user):
        s1 = re.search(r"SUMMARY 1:\n(.+?)\n\nSUMMARY 2", user, flags=re.S).group(1)
        s2 = re.search(r"SUMMARY 2:\n(.+?)$", user, flags=re.S).group(1)
        def score(s, position):
            return (2.0 if "12,480" in s else 0) + len(s) / 150 + (1.1 if position == 1 else 0)
        best = 1 if score(s1, 1) >= score(s2, 2) else 2
        return json.dumps({"best": best, "justification": "More complete and better presented."}, ensure_ascii=False)

    def _rag(self, prompt):
        if "12,480" in prompt:
            return "The LFS 2025 sample covers 12,480 households [P3]."
        return "The 2025 LFS of Fictivia covers a sample of roughly 10,000 households."

    def respond(self, messages, temperature=0.0, json_mode=False):
        prompt = "\n".join(m["content"] for m in messages)
        system = messages[0]["content"] if messages[0]["role"] == "system" else ""
        rng = random.Random() if temperature > 0 else random.Random(0)
        if "Fictivia" in prompt:
            text = self._rag(prompt)
        elif "BULLETIN" in prompt:
            text = self._needle(prompt)
        elif "EVALUATOR" in system:
            text = self._judge(messages[-1]["content"])
        elif "ISCO" in prompt or "occupation" in prompt.lower():
            text = self._coding(messages, prompt, temperature, rng)
        else:
            text = "Simulated answer."
        t_in, t_out = count_tokens(prompt) + 4 * len(messages), count_tokens(text)
        latency = 0.12 + 0.004 * t_out + 0.00004 * t_in
        return text, t_in, t_out, 0, latency

SIMULATOR = LLMSimulator()
print("✅ Simulator ready.")

In [ ]:
# 🔌 Unified LLM client: measurement, caching, call log, retries
class LLMClient:
    URLS = {"groq": "https://api.groq.com/openai/v1", "ollama": "http://localhost:11434/v1"}

    def __init__(self, provider, model, key=None, base_url=None):
        self.provider, self.model = provider, model
        self.cache, self.calls = {}, []
        self._client = None
        if provider != "simulation":
            from openai import OpenAI
            url = base_url or self.URLS.get(provider)
            self._client = OpenAI(api_key=key or "local-key", base_url=url)

    def _cache_key(self, messages, temperature, json_mode, max_tokens):
        raw = json.dumps([self.provider, self.model, messages, temperature, json_mode, max_tokens],
                         ensure_ascii=False, sort_keys=True)
        return hashlib.sha256(raw.encode()).hexdigest()

    def _api_call(self, messages, temperature, max_tokens, json_mode):
        args = dict(model=self.model, messages=messages, temperature=temperature, max_tokens=max_tokens)
        if json_mode:
            args["response_format"] = {"type": "json_object"}
        if self.provider == "groq" and "gpt-oss" in self.model:
            args["extra_body"] = {"reasoning_effort": "low"}
        for attempt in range(6):
            try:
                response = self._client.chat.completions.create(**args)
                break
            except Exception as error:
                message = str(error).lower()
                if "reasoning" in message and "extra_body" in args:
                    args.pop("extra_body"); continue
                if "response_format" in message and "response_format" in args:
                    args.pop("response_format"); continue
                if attempt < 5 and any(k in message for k in ["rate", "429", "timeout", "503", "overloaded", "connection"]):
                    wait = 2 ** attempt * 2
                    print(f"   ⏳ Rate limit or temporary error: retrying in {wait} s")
                    time.sleep(wait); continue
                raise
        text = response.choices[0].message.content or ""
        usage = response.usage
        details = getattr(usage, "prompt_tokens_details", None)
        cached = (getattr(details, "cached_tokens", 0) or 0) if details else 0
        if PAUSE_BETWEEN_CALLS_S:
            time.sleep(PAUSE_BETWEEN_CALLS_S)
        return text, usage.prompt_tokens or 0, usage.completion_tokens or 0, cached

    def chat(self, messages, temperature=0.0, max_tokens=1024, json_mode=False, use_cache=True, label=""):
        key = self._cache_key(messages, temperature, json_mode, max_tokens)
        if use_cache and temperature == 0 and key in self.cache:
            r = dict(self.cache[key], from_cache=True, latency_s=0.0, input_tokens=0, output_tokens=0,
                     provider_cached_tokens=0, cost_usd=0.0)
            self.calls.append(dict(r, label=label, text=None))
            return r
        start = time.perf_counter()
        if self.provider == "simulation":
            text, t_in, t_out, cached, latency = SIMULATOR.respond(messages, temperature, json_mode)
        else:
            text, t_in, t_out, cached = self._api_call(messages, temperature, max_tokens, json_mode)
            latency = time.perf_counter() - start
        r = dict(text=text, input_tokens=t_in, output_tokens=t_out, provider_cached_tokens=cached,
                 latency_s=round(latency, 3), cost_usd=usd_cost(t_in, t_out), from_cache=False)
        if temperature == 0:
            self.cache[key] = r
        self.calls.append(dict(r, label=label, text=None))
        return r

    def summary(self, label=None):
        df = pd.DataFrame(self.calls)
        if df.empty:
            return df
        if label:
            df = df[df["label"] == label]
        return (df.groupby("label")
                  .agg(calls=("label", "size"), from_cache=("from_cache", "sum"),
                       input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"),
                       cost_usd=("cost_usd", "sum"), mean_latency_s=("latency_s", "mean"))
                  .round(4))

LLM = LLMClient(ACTIVE_PROVIDER, ACTIVE_MODEL,
                key=GROQ_KEY if ACTIVE_PROVIDER == "groq" else OPENAI_KEY,
                base_url=OPENAI_COMPATIBLE_URL if ACTIVE_PROVIDER == "openai" else None)

test = LLM.chat([{"role": "user", "content": "Answer with one word: OK."}], label="test")
print("✅ Client ready · Test answer:", test["text"][:80], "·", test["input_tokens"], "input tokens")

In [ ]:
# 📋 (Optional, live mode) List the active models at your provider
if not SIMULATION_MODE:
    try:
        models = sorted(m.id for m in LLM._client.models.list().data)
        print(f"{len(models)} models available:")
        print(" · ".join(models))
    except Exception as e:
        print("Could not list models:", e)
else:
    print("Simulation mode: no model list to show.")

<div style="background:#00A86A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">01</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">From good to great</span><br>
<i style="color:#E6F6EE">Why does a prompt that worked on yesterday’s bulletin fail on today’s?</i>
</div>

### 1.1 The experiment everybody runs

You write a prompt, try it on **one** example, the answer looks right… and you conclude that “it works”. Let us do exactly that.

In [ ]:
naive_prompt = "What is the ISCO code for this occupation: {text}"

easy_example = "Vendeuse de tomates au marché central"
r = LLM.chat([{"role": "user", "content": naive_prompt.format(text=easy_example)}], label="section1")
print("Input   :", easy_example)
print("Answer  :", r["text"])
print("Expected: 5211 (stall and market salespersons)")

The answer **looks** plausible. But look closely: can a machine use it? Does the code it quotes actually exist in ISCO-08? Let us now try a few real descriptions, as enumerators type them.

In [ ]:
a_few_cases = [
    ("Vend des cartes de recharge dans la rue", "9520"),
    ("Primary teacher, sells airtime at weekends", "2341"),
    ("Ingénieur", "UNCODABLE"),
    ("Moto-taxi driver, owns the motorcycle", "8321"),
]
for text, expected in a_few_cases:
    r = LLM.chat([{"role": "user", "content": naive_prompt.format(text=text)}], label="section1")
    print(f"• {text}\n  expected: {expected}\n  answer  : {r['text'][:160]}\n")

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
<b>📌 What we observe</b> (in simulation, and with most real models too):
<ul>
<li>answers in <b>prose</b>, which no production pipeline can ingest automatically;</li>
<li><b>invented</b> codes, or codes at the wrong level of detail, because no reference list was given;</li>
<li>a code assigned to “Ingénieur” even though the information is <b>insufficient</b>;</li>
<li>confusion when a person has <b>two activities</b>.</li>
</ul>
One example proves nothing. For official statistics, the <b>UN Fundamental Principles</b> (Resolution 68/261, 2014) require methods chosen on professional and scientific grounds (Principle 2). An LLM step in production must meet the same test: <b>a measured error rate</b>.
</div>

| | A “good” prompt | A “great” prompt |
|---|---|---|
| **Tested on** | the example you happened to try | a frozen set of 30–50 representative cases |
| **Judged by** | a favourable impression | written criteria and a numeric score |
| **Error rate** | unknown | measured, with named failure modes |
| **Reproducibility** | answers drift | pinned model, versioned prompt |
| **Cost** | discovered on the invoice | estimated per 1,000 documents in advance |

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">02</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">The evaluation set</span><br>
<i style="color:#E6F6EE">How will you know that version 2 is better than version 1?</i>
</div>

### 2.1 What an evaluation set is

An **evaluation set** is a **frozen** collection of cases, each with a **gold answer** agreed by experts. For a prompt, it plays the role of a control sample in manual coding.

We use three **separate** sets:

| Set | Role | Size here |
|---|---|---|
| 🟢 **Example pool** | cases you are allowed to copy into the prompt | 6 |
| 🔵 **Evaluation set** | cases on which every version is measured | 30 |
| 🟠 **Holdout set** | cases consulted rarely, to confirm real gains | 8 |

And three **case types**:
- **typical** (~60 %): the everyday path, in every language;
- **edge** (~25 %): where decision rules are tested (two jobs, neighbouring categories);
- **negative** (~15 %): the model must answer `UNCODABLE` instead of inventing a code.

In [ ]:
# 📚 Reference list (abridged ISCO-08 unit group titles, ILO 2012)
ISCO_LIST = {
    "2211": "Generalist medical practitioners", "2221": "Nursing professionals",
    "2330": "Secondary education teachers", "2341": "Primary school teachers",
    "2411": "Accountants", "2512": "Software developers", "4110": "General office clerks",
    "4132": "Data entry clerks", "5120": "Cooks", "5141": "Hairdressers",
    "5211": "Stall and market salespersons", "5212": "Street food salespersons",
    "5223": "Shop sales assistants", "5311": "Child care workers",
    "5414": "Security guards", "6111": "Field crop and vegetable growers",
    "6121": "Livestock and dairy producers", "7112": "Bricklayers and related workers",
    "7115": "Carpenters and joiners", "7231": "Motor vehicle mechanics and repairers",
    "7411": "Building and related electricians", "7512": "Bakers, pastry-cooks and confectionery makers",
    "7531": "Tailors, dressmakers, furriers and hatters", "8321": "Motorcycle drivers",
    "8322": "Car, taxi and van drivers", "9112": "Cleaners and helpers in offices and hotels",
    "9520": "Street vendors (excluding food)",
}
LIST_TEXT = "\n".join(f"- {c}: {t}" for c, t in ISCO_LIST.items())

# 🟢 Example pool (may be used inside the prompt)
EXAMPLE_POOL = [
    ("Vendeur de poisson au marché", "5211"),
    ("Conduit une moto pour transporter des passagers", "8321"),
    ("Teacher at primary school, also farms on weekends", "2341"),
    ("Vend des arachides grillées dans la rue", "5212"),
    ("Fonctionnaire", "UNCODABLE"),
    ("Sells phone credit on the street", "9520"),
]

# 🔵 Evaluation set: 30 frozen cases, as typed by enumerators (French and English)
_cases = [
    ("E01", "Vendeuse de tomates au marché central", "fr", "5211", "typical"),
    ("E02", "Moto-taxi driver, owns the motorcycle", "en", "8321", "typical"),
    ("E03", "Agent de saisie à l'institut de statistique", "fr", "4132", "typical"),
    ("E04", "Cultive du maïs et du manioc sur son champ", "fr", "6111", "typical"),
    ("E05", "Taxi driver in the capital", "en", "8322", "typical"),
    ("E06", "Couturière dans un atelier de quartier", "fr", "7531", "typical"),
    ("E07", "Maçon sur les chantiers de construction", "fr", "7112", "typical"),
    ("E08", "Hairdresser in a beauty salon", "en", "5141", "typical"),
    ("E09", "Infirmière diplômée à l'hôpital régional", "fr", "2221", "typical"),
    ("E10", "Software developer for a mobile banking company", "en", "2512", "typical"),
    ("E11", "Gardien de nuit dans une banque", "fr", "5414", "typical"),
    ("E12", "Mécanicien automobile dans un garage", "fr", "7231", "typical"),
    ("E13", "Boulanger, fabrique le pain chaque matin", "fr", "7512", "typical"),
    ("E14", "Cook in a hotel restaurant", "en", "5120", "typical"),
    ("E15", "Comptable dans une PME", "fr", "2411", "typical"),
    ("E16", "Élève des chèvres et des vaches", "fr", "6121", "typical"),
    ("E17", "Électricien, installe le câblage des maisons", "fr", "7411", "typical"),
    ("E18", "Cleaner in government offices", "en", "9112", "typical"),
    ("L01", "Primary teacher, sells airtime at weekends", "en", "2341", "edge"),
    ("L02", "Vend des beignets au bord de la route", "fr", "5212", "edge"),
    ("L03", "Vend des cartes de recharge dans la rue", "fr", "9520", "edge"),
    ("L04", "Enseignante au lycée, donne des cours particuliers le soir", "fr", "2330", "edge"),
    ("L05", "Nounou, garde les enfants des voisins", "fr", "5311", "edge"),
    ("L06", "Shop assistant in a supermarket", "en", "5223", "edge"),
    ("L07", "Agent administratif à la mairie", "fr", "4110", "edge"),
    ("L08", "Docteur au centre de santé", "fr", "2211", "edge"),
    ("N01", "Ingénieur", "fr", "UNCODABLE", "negative"),
    ("N02", "works", "en", "UNCODABLE", "negative"),
    ("N03", "Aide ses parents", "fr", "UNCODABLE", "negative"),
    ("N04", "Business", "en", "UNCODABLE", "negative"),
]
EVAL_SET = pd.DataFrame(_cases, columns=["id", "text", "language", "gold_code", "case_type"])

# 🟠 Holdout set: looked at only occasionally
HOLDOUT_SET = pd.DataFrame([
    ("H01", "Vendeur de pagnes au grand marché", "fr", "5211", "typical"),
    ("H02", "Conducteur de taxi-moto", "fr", "8321", "typical"),
    ("H03", "Institutrice, vend aussi des beignets le week-end", "fr", "2341", "edge"),
    ("H04", "Vend des cacahuètes grillées au bord de la route", "fr", "5212", "edge"),
    ("H05", "Nurse at a private clinic", "en", "2221", "typical"),
    ("H06", "Travailleur", "fr", "UNCODABLE", "negative"),
    ("H07", "Vendeuse dans un supermarché", "fr", "5223", "edge"),
    ("H08", "Mechanic repairing cars and trucks", "en", "7231", "typical"),
], columns=["id", "text", "language", "gold_code", "case_type"])

# 🔒 "Freeze" the set: its fingerprint changes if a single character is edited
def fingerprint(df):
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:12]

EVAL_FINGERPRINT = fingerprint(EVAL_SET)
print(f"Evaluation set: {len(EVAL_SET)} cases · fingerprint {EVAL_FINGERPRINT}")
print(f"Holdout set   : {len(HOLDOUT_SET)} cases · fingerprint {fingerprint(HOLDOUT_SET)}")
display(EVAL_SET.head(8))

In [ ]:
# 📊 Is the set representative? Composition by case type and by language
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
types = EVAL_SET["case_type"].value_counts().reindex(["typical", "edge", "negative"])
axes[0].barh(types.index, types.values, color=[GREEN, OCHRE, BRICK])
for i, v in enumerate(types.values):
    axes[0].text(v + 0.3, i, f"{v} ({v/len(EVAL_SET):.0%})", va="center", color=INK)
axes[0].set_title("Case types"); axes[0].invert_yaxis(); axes[0].set_xlim(0, 23)
languages = EVAL_SET["language"].value_counts()
axes[1].bar(languages.index.str.upper(), languages.values, color=[DEEP_GREEN, TEAL])
for i, v in enumerate(languages.values):
    axes[1].text(i, v + 0.3, str(v), ha="center", color=INK)
axes[1].set_title("Languages of the answers"); axes[1].set_ylim(0, 23)
plt.tight_layout(); plt.show()

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
<b>📌 Five rules for an evaluation set you can trust</b>
<ol>
<li><b>Representative</b>: every language, region, source and format met in production, including typos.</li>
<li><b>Expert-labelled</b>: double coding, disagreements resolved before scoring starts.</li>
<li><b>Frozen and versioned</b>: never edited once scoring has started (hence the fingerprint above), stored in Git.</li>
<li><b>Never in the prompt</b>: prompt examples come from a separate pool (section 3.5 shows what happens otherwise).</li>
<li><b>Grown from failures</b>: every error met in production becomes a new case.</li>
</ol>
<i>Rule of thumb: start with 30 to 50 cases.</i>
</div>

<div style="background:#0E7C86;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">03</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Measuring quality</span><br>
<i style="color:#E6F6EE">Score it, don’t sense it.</i>
</div>

### 3.1 Choose the metric that matches the task

| Task type | Example in an NSO | Metric | Scoring |
|---|---|---|---|
| Extraction | CPI figures from a PDF bulletin | field-level exact match | automatic |
| **Coding** ⬅️ *our case* | ISCO, ISIC, COICOP | **accuracy**, per-class precision / recall | automatic + confusion matrix |
| Structured output | JSON feeding a dashboard | **valid JSON rate** | automatic |
| Grounded Q&A (RAG) | methodology notes | faithfulness, citation accuracy | rubric, human or calibrated judge |
| Drafting | press release | rubric 1–5 | reviewers, calibrated judge |

For our task we measure **two accuracies**:
- **strict**: the answer must be valid JSON containing the right code. This is what matters in production, because an answer a machine cannot read is a failure;
- **lenient**: we look for a code anywhere in the text. It shows what the model “knows”, independently of the format.

In [ ]:
# 🧮 Evaluation engine
LOG = []   # iteration log: one row per version tested

def extract_code(text):
    """Return (strict_code, lenient_code, valid_json)."""
    raw = re.sub(r"^```(?:json)?|```$", "", (text or "").strip(), flags=re.M).strip()
    strict_code, valid_json = None, False
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict) and obj.get("isco_code"):
            strict_code, valid_json = str(obj["isco_code"]).strip().upper(), True
    except Exception:
        pass
    m = re.search(r"\b(\d{4}|UNCODABLE)\b", text or "")
    return strict_code, (m.group(1) if m else None), valid_json

def evaluate(version, build_messages, dataset=None, json_mode=False, temperature=0.0,
             change="", log_it=True, show=True, use_cache=True):
    dataset = EVAL_SET if dataset is None else dataset
    rows = []
    for case in dataset.itertuples():
        r = LLM.chat(build_messages(case.text), temperature=temperature, json_mode=json_mode,
                     label=version, use_cache=use_cache)
        strict, lenient, valid = extract_code(r["text"])
        rows.append(dict(id=case.id, text=case.text, case_type=case.case_type, gold_code=case.gold_code,
                         predicted_code=strict or lenient, valid_json=valid,
                         correct_strict=(strict == case.gold_code), correct_lenient=(lenient == case.gold_code),
                         input_tokens=r["input_tokens"], output_tokens=r["output_tokens"],
                         cost_usd=r["cost_usd"], answer=r["text"]))
    df = pd.DataFrame(rows)
    summary = dict(version=version, change=change, dataset="holdout" if dataset is HOLDOUT_SET else "evaluation",
                   strict_accuracy=df["correct_strict"].mean(), lenient_accuracy=df["correct_lenient"].mean(),
                   valid_json=df["valid_json"].mean(),
                   tokens_per_case=df["input_tokens"].sum() / len(df) if df["input_tokens"].sum() else None,
                   cost_per_1000_usd=df["cost_usd"].sum() / len(df) * 1000 if df["cost_usd"].sum() else None,
                   decision="")
    for t in ["typical", "edge", "negative"]:
        subset = df[df["case_type"] == t]
        summary[f"acc_{t}"] = subset["correct_strict"].mean() if len(subset) else None
    if log_it:
        LOG[:] = [j for j in LOG if not (j["version"] == version and j["dataset"] == summary["dataset"])]
        LOG.append(summary)
    if show:
        score_card(summary)
    return df, summary

def score_card(res):
    def tile(title, value, colour):
        return (f'<div style="flex:1;background:#F4F7F5;border:1px solid {SAGE};border-radius:10px;padding:10px 14px;margin:4px">'
                f'<div style="font-size:11px;letter-spacing:1px;color:{SLATE};font-weight:700">{title}</div>'
                f'<div style="font-size:26px;font-weight:800;color:{colour}">{value}</div></div>')
    pct = lambda x: "—" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.0%}"
    cost = "—" if not res["cost_per_1000_usd"] else f"${res['cost_per_1000_usd']:.3f}"
    html = (f'<div style="font-weight:800;color:{DEEP_GREEN};margin-top:6px">VERSION {res["version"]} · {res["dataset"]} set'
            f'<span style="font-weight:400;color:{SLATE}"> · {res["change"]}</span></div>'
            f'<div style="display:flex;flex-wrap:wrap">'
            + tile("STRICT ACCURACY", pct(res["strict_accuracy"]), DEEP_GREEN)
            + tile("LENIENT ACCURACY", pct(res["lenient_accuracy"]), TEAL)
            + tile("VALID JSON", pct(res["valid_json"]), OCHRE)
            + tile("TYPICAL · EDGE · NEGATIVE",
                   f'{pct(res["acc_typical"])} · {pct(res["acc_edge"])} · {pct(res["acc_negative"])}', INK)
            + tile("COST / 1,000 CASES", cost, TERRA) + "</div>")
    display(HTML(html))

def decide(version, decision):
    for j in LOG:
        if j["version"] == version:
            j["decision"] = decision

def show_errors(df, n=10):
    errors = df[~df["correct_strict"]][["id", "case_type", "text", "gold_code", "predicted_code", "valid_json"]]
    if errors.empty:
        callout("No errors on this set.", "note")
    else:
        display(errors.head(n))

print("✅ Evaluation engine ready.")

### 3.2 The optimisation loop

<div style="display:flex;flex-wrap:wrap;gap:6px;margin:8px 0">
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">1 · Define success</b><br><small>criteria and target score</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">2 · Build the set</b><br><small>done in section 2</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">3 · Run the baseline</b><br><small>score of v0</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">4 · Diagnose</b><br><small>read the failures</small></div>
<div style="flex:1;min-width:120px;background:#E8F5EF;border:2px solid #D49A00;border-radius:8px;padding:8px"><b style="color:#D49A00">5 · Change ONE thing</b><br><small>a rule, an example or a setting</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">6 · Re-test and log</b><br><small>iteration log</small></div>
</div>

**Step 1: define success before writing a single prompt.**

In [ ]:
TARGET_ACCURACY = 0.90   # minimum strict accuracy on the evaluation set
TARGET_JSON = 1.00       # every answer must be machine-readable
print(f"🎯 Target: strict accuracy ≥ {TARGET_ACCURACY:.0%} and valid JSON = {TARGET_JSON:.0%}")

### 3.3 Version 0: the baseline

The naive prompt from section 1, measured on all 30 cases. This is our **reference point**.

In [ ]:
def messages_v0(text):
    return [{"role": "user", "content": f"What is the ISCO code for this occupation: {text}"}]

df_v0, res_v0 = evaluate("v0", messages_v0, change="naive prompt (baseline)")
decide("v0", "baseline")
show_errors(df_v0, 6)

**Step 4: diagnose.** Three causes stand out: (a) answers in prose, so valid JSON is 0 % and strict accuracy is nil; (b) invented codes, because no list was supplied; (c) no instruction for vague cases or multiple jobs.

**Step 5: change one thing only.** Start with the most structural cause: give the model a **role**, an **objective**, the **context** and the **reference list**.

### 3.4 Versions 1 to 4: one change at a time

In [ ]:
SYSTEM_V1 = f"""You are a statistical coder at the National Institute of Statistics.
OBJECTIVE: assign to each job description the most appropriate ISCO-08 four-digit unit group code.
CONTEXT: the descriptions come from the labour force survey. They are written in French or English by enumerators, often in abbreviated form.
Use only the codes from the following list:
{LIST_TEXT}"""

def messages_v1(text):
    return [{"role": "system", "content": SYSTEM_V1},
            {"role": "user", "content": f"Description: {text}"}]

df_v1, res_v1 = evaluate("v1", messages_v1, change="+ role, objective, context, code list")

Lenient accuracy improves (fewer invented codes), but strict accuracy is still nil: the answer is not yet machine-readable. Next change: **impose the format**. We also switch on the API's JSON mode (`json_mode=True`), which is a settings-side lever.

In [ ]:
JSON_FORMAT = """
OUTPUT FORMAT
Answer with a JSON object only, with no text around it:
{"isco_code": "<four-digit code>", "confidence": "high|medium|low"}"""

SYSTEM_V2 = SYSTEM_V1 + "\n" + JSON_FORMAT

def messages_v2(text):
    return [{"role": "system", "content": SYSTEM_V2},
            {"role": "user", "content": f"Description: {text}"}]

df_v2, res_v2 = evaluate("v2", messages_v2, json_mode=True, change="+ JSON format imposed")
show_errors(df_v2, 10)

The format is fixed. The remaining errors concentrate on **negative cases** (guessed codes) and **multiple jobs**. Next change: write explicit **decision rules**.

In [ ]:
RULES = """
RULES
1. If the person has several activities, code only the main job (the one mentioned first).
2. If the description is too vague to determine a four-digit unit group, answer "UNCODABLE" in the isco_code field. Never guess."""

SYSTEM_V3 = SYSTEM_V1 + "\n" + RULES + "\n" + JSON_FORMAT

def messages_v3(text):
    return [{"role": "system", "content": SYSTEM_V3},
            {"role": "user", "content": f"Description: {text}"}]

df_v3, res_v3 = evaluate("v3", messages_v3, json_mode=True, change="+ rules: main job, UNCODABLE")
show_errors(df_v3, 10)

Confusions remain between **neighbouring categories**: market stalls (5211), street food (5212) and other street vending (9520). A written rule would work; here we test the other lever: **showing examples**, drawn from the pool and never from the evaluation set.

In [ ]:
EXAMPLES_TEXT = "\nEXAMPLES\n" + "\n".join(f'- "{t}" → {c}' for t, c in EXAMPLE_POOL)
SYSTEM_V4 = SYSTEM_V1 + "\n" + RULES + "\n" + EXAMPLES_TEXT + "\n" + JSON_FORMAT

def messages_v4(text):
    return [{"role": "system", "content": SYSTEM_V4},
            {"role": "user", "content": f"Description: {text}"}]

df_v4, res_v4 = evaluate("v4", messages_v4, json_mode=True, change="+ 6 examples from the pool")
target_v4 = bool(res_v4["strict_accuracy"] >= TARGET_ACCURACY)
callout(f"Target of {TARGET_ACCURACY:.0%} {'reached' if target_v4 else '<b>not reached yet</b>'} "
        f"with v4 ({res_v4['strict_accuracy']:.0%}). "
        + ("" if target_v4 else "You will try to beat it yourself in section 7."), "note" if target_v4 else "warning")
show_errors(df_v4, 10)
print("\nSystem prompt v4:\n" + "─" * 60 + "\n" + SYSTEM_V4)

### 3.5 The leakage trap: “improving” by copying the test

A classic temptation: copy the failing evaluation cases into the prompt. The score rises… but has the prompt really improved? Let us check on the **holdout set**, which the prompt has never seen.

In [ ]:
leak_ids = set(df_v4.loc[~df_v4["correct_strict"], "id"]) | set(EVAL_SET.loc[EVAL_SET["case_type"] != "typical", "id"])
leak_cases = EVAL_SET[EVAL_SET["id"].isin(leak_ids)]   # the failing cases AND the hard ones
LEAK_TEXT = EXAMPLES_TEXT + "\n" + "\n".join(f'- "{t}" → {c}' for t, c in zip(leak_cases["text"], leak_cases["gold_code"]))
SYSTEM_V5 = SYSTEM_V1 + "\n" + RULES + "\n" + LEAK_TEXT + "\n" + JSON_FORMAT

def messages_v5(text):
    return [{"role": "system", "content": SYSTEM_V5},
            {"role": "user", "content": f"Description: {text}"}]

df_v5, res_v5 = evaluate("v5", messages_v5, json_mode=True, change="+ evaluation cases copied in (LEAKAGE)")
print("Control on the holdout set:")
_, res_v4_h = evaluate("v4", messages_v4, dataset=HOLDOUT_SET, json_mode=True, change="control")
_, res_v5_h = evaluate("v5", messages_v5, dataset=HOLDOUT_SET, json_mode=True, change="control")

In [ ]:
comparison = pd.DataFrame({
    "Evaluation set": [res_v4["strict_accuracy"], res_v5["strict_accuracy"]],
    "Holdout set": [res_v4_h["strict_accuracy"], res_v5_h["strict_accuracy"]],
}, index=["v4 (examples from the pool)", "v5 (test cases copied in)"])
ax = comparison.plot.bar(color=[DEEP_GREEN, OCHRE], figsize=(8, 3.4), rot=0, width=0.65)
for c in ax.containers:
    ax.bar_label(c, labels=[f"{v:.0%}" for v in c.datavalues], padding=2)
ax.set_ylim(0, 1.15); ax.set_yticks([]); ax.set_title("Leakage inflates the score… without any real gain")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout(); plt.show()
decide("v5", "rejected: leakage")
callout("<b>Lesson:</b> v5 “wins” on the evaluation set, but not on the holdout set. "
        "It memorised the test. Prompt examples must always come from a separate pool.", "risk")

### 3.6 The iteration log

The log makes your prompt **auditable**: one version, one change, a score, a decision.

In [ ]:
decide("v1", "kept"); decide("v2", "kept"); decide("v3", "kept"); decide("v4", "kept")

def show_log():
    j = pd.DataFrame([x for x in LOG if x["dataset"] == "evaluation"])
    columns = ["version", "change", "strict_accuracy", "lenient_accuracy", "valid_json",
               "acc_typical", "acc_edge", "acc_negative", "tokens_per_case", "decision"]
    view = j[columns].copy()
    for c in ["strict_accuracy", "lenient_accuracy", "valid_json", "acc_typical", "acc_edge", "acc_negative"]:
        view[c] = view[c].map(lambda x: f"{x:.0%}")
    display(view.set_index("version"))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    x = range(len(j))
    colours = [BRICK if str(d).startswith("reject") else DEEP_GREEN for d in j["decision"]]
    ax.bar([i - 0.2 for i in x], j["strict_accuracy"], 0.4, color=colours, label="Strict accuracy")
    ax.bar([i + 0.2 for i in x], j["valid_json"], 0.4, color=RAMP[0], label="Valid JSON")
    for i, v in enumerate(j["strict_accuracy"]):
        ax.text(i - 0.2, v + 0.02, f"{v:.0%}", ha="center", fontsize=9, color=INK)
    ax.axhline(TARGET_ACCURACY, color=GOLD, ls="--", lw=1.5)
    ax.text(-0.45, TARGET_ACCURACY + 0.02, f"target {TARGET_ACCURACY:.0%}", color=OCHRE, ha="left", fontweight="bold")
    ax.set_xticks(list(x)); ax.set_ylim(0, 1.12); ax.set_yticks([])
    ax.set_xticklabels([f"{v}\n{'rejected' if str(d).startswith('reject') else ''}" for v, d in zip(j["version"], j["decision"])])
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
    ax.set_title("Version history on the evaluation set (red = rejected version)")
    plt.tight_layout(); plt.show()

show_log()

### 3.7 Beyond the average: per-class analysis

A high overall accuracy can hide a category that is **systematically** mis-coded. Let us look at the confusion matrix of the best legitimate version (v4).

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(df_v4["gold_code"], df_v4["predicted_code"].fillna("NONE"),
                               output_dict=True, zero_division=0)
per_class = (pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
             .query("support > 0")[["precision", "recall", "support"]])
display(per_class.sort_values("recall").head(8).style.format({"precision": "{:.0%}", "recall": "{:.0%}", "support": "{:.0f}"})
        .background_gradient(subset=["recall"], cmap="RdYlGn", vmin=0, vmax=1))

confusions = df_v4[df_v4["gold_code"] != df_v4["predicted_code"]]
if len(confusions):
    display(pd.crosstab(confusions["gold_code"], confusions["predicted_code"].fillna("NONE"))
            .rename_axis(index="expected", columns="predicted"))
callout("The remaining confusions tell you <b>where</b> to act: a targeted rule or an extra example "
        "for the category concerned. You will do this yourself in section 7.", "note")

<div style="background:#0E7C86;color:#FFFFFF;padding:10px 18px;border-radius:8px">
<b>3.8 Aside: using an LLM as a judge</b>
</div>

For **drafting** or **question answering**, no script can score automatically. A second model, the **judge**, can do it against a written rubric. But judges have **known biases** (Zheng et al., 2023): they often favour the answer shown **first**, **longer** answers, and answers written in **their own style**.

A simple **position-bias test**: present the same pair in both orders. A reliable judge picks the same summary.

In [ ]:
JUDGE_SOURCE = ("Methodological note: the 2025 LFS interviewed 12,480 households between February and April, "
                "with a response rate of 91 %. Results are representative at regional level.")
PAIRS = [
    ("The 2025 LFS covers 12,480 households (February-April, 91 % response), representative by region.",
     "The 2025 labour force survey, carried out during the first half of the year, interviewed a large sample of "
     "households across all regions of the country, with an excellent response rate, which guarantees robust "
     "results that are useful for public policy."),
    ("Survey of 12,480 households, response rate 91 %.",
     "The 2025 LFS covers 12,480 households interviewed from February to April; the response rate reaches 91 % "
     "and results are representative at regional level."),
]
JUDGE_SYSTEM = ("You are an EVALUATOR of statistical summaries. Compare two summaries of the same source against this rubric: "
                "accuracy of figures (priority), completeness, concision. "
                'Answer in JSON: {"best": 1 or 2, "justification": "..."}')

def judge(source, s1, s2):
    content = f"SOURCE:\n{source}\n\nSUMMARY 1:\n{s1}\n\nSUMMARY 2:\n{s2}"
    r = LLM.chat([{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": content}],
                 json_mode=True, label="judge")
    try:
        return int(json.loads(r["text"])["best"])
    except Exception:
        return None

rows = []
for i, (a, b) in enumerate(PAIRS, 1):
    order_ab = judge(JUDGE_SOURCE, a, b)            # 1 = A
    order_ba = judge(JUDGE_SOURCE, b, a)            # 2 = A
    choice_ab = "A" if order_ab == 1 else "B"
    choice_ba = "A" if order_ba == 2 else "B"
    rows.append(dict(pair=i, choice_order_AB=choice_ab, choice_order_BA=choice_ba,
                     consistent="✅" if choice_ab == choice_ba else "❌ position bias"))
display(pd.DataFrame(rows))
callout("<b>Calibrate before you trust:</b> have humans score 20 to 30 outputs, compare with the judge, "
        "present each pair in both orders, and use the judge only if agreement is high.", "warning")

<div style="background:#D49A00;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #00553A">
<span style="color:#FFFFFF;font-size:30px;font-weight:800">04</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Controlling variance</span><br>
<i style="color:#FFF7E0">Same prompt, same input: why a different answer?</i>
</div>

### 4.1 Where variance comes from

| Source | Explanation | Lever |
|---|---|---|
| 🎲 **Sampling** | the model picks each token with some randomness, set by **temperature** | low temperature (0 to 0.2) for extraction and coding |
| ❓ **Ambiguous instructions** | an unclear rule is settled differently on each run | explicit tie-break rules |
| 🔄 **Silent model updates** | a generic alias can point to a new version | pin the exact model identifier |
| 📄 **Input variation** | language, layout, typos | a representative evaluation set |

### 4.2 The five-run consistency test

We run each case **five times** and measure the **agreement rate**: the share of runs giving the most frequent answer. The cache is switched off here; otherwise repeated runs at temperature 0 would be identical by construction.

In [ ]:
SUBSET = EVAL_SET[EVAL_SET["id"].isin(["E01", "E02", "E05", "E13", "L01", "L02", "L03", "L04", "L06", "N01"])]
N_RUNS = 5

def consistency_test(build_messages, temperature, label):
    rows = []
    for case in SUBSET.itertuples():
        codes = []
        for _ in range(N_RUNS):
            r = LLM.chat(build_messages(case.text), temperature=temperature, json_mode=True,
                         use_cache=False, label=label)
            strict, lenient, _ = extract_code(r["text"])
            codes.append(strict or lenient or "UNREADABLE")
        majority, n = Counter(codes).most_common(1)[0]
        rows.append(dict(id=case.id, text=case.text[:38], gold_code=case.gold_code, answers=" ".join(codes),
                         agreement=n / N_RUNS, correct_first_run=codes[0] == case.gold_code,
                         correct_vote=majority == case.gold_code))
    return pd.DataFrame(rows)

cons_t0 = consistency_test(messages_v3, 0.0, "variance_t0")
cons_t1 = consistency_test(messages_v3, 1.0, "variance_t1")
display(cons_t1[["id", "text", "gold_code", "answers", "agreement"]].style.format({"agreement": "{:.0%}"})
        .background_gradient(subset=["agreement"], cmap="RdYlGn", vmin=0.4, vmax=1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
x = range(len(cons_t0))
ax.bar([i - 0.2 for i in x], cons_t0["agreement"], 0.4, color=DEEP_GREEN, label="temperature 0")
ax.bar([i + 0.2 for i in x], cons_t1["agreement"], 0.4, color=OCHRE, label="temperature 1")
ax.set_xticks(list(x)); ax.set_xticklabels(cons_t0["id"]); ax.set_ylim(0, 1.15)
ax.set_ylabel("agreement over 5 runs"); ax.legend(frameon=False, ncol=2, loc="upper left")
ax.set_title("Edge cases (L..) are the most unstable when temperature rises")
plt.tight_layout(); plt.show()
print(f"Mean agreement · temperature 0: {cons_t0['agreement'].mean():.0%} · temperature 1: {cons_t1['agreement'].mean():.0%}")

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px">
⚠️ <b>Temperature 0 reduces randomness; it does not guarantee identical outputs</b> at every provider (in simulation, agreement at temperature 0 is perfect; that is not always true with a real model). <b>Always measure</b> agreement on your own set.
</div>

### 4.3 Majority voting (self-consistency)

For high-stakes cases you can run several times and keep the **majority answer** (Wang et al., 2022). It costs more, but **disagreement between runs** is also an excellent signal for **routing a case to a human coder**.

In [ ]:
recap = pd.DataFrame({
    "Accuracy, single run": [cons_t1["correct_first_run"].mean()],
    "Accuracy, vote over 5": [cons_t1["correct_vote"].mean()],
    "Cases to review (agreement < 100 %)": [(cons_t1["agreement"] < 1).sum()],
    "Relative cost": ["× 5"],
}, index=["temperature 1"])
display(recap.style.format({"Accuracy, single run": "{:.0%}", "Accuracy, vote over 5": "{:.0%}"}))
to_review = cons_t1[cons_t1["agreement"] < 1]["id"].tolist()
callout(f"Cases to route to a human coder: <b>{', '.join(to_review) or 'none'}</b>. "
        "Disagreement between runs is a free confidence indicator.", "remember")

### 4.4 Six levers for reproducible outputs

| Model settings | Prompt design |
|---|---|
| 🌡️ **Low temperature** (0 to 0.2) for extraction and coding | 🧾 **Constrain the format**: JSON schema, closed list of values |
| 🏷️ **Pin the model version** and record it in the log | ⚖️ **Tie-break rules**: “if two jobs, code the main one” |
| ✂️ **Cap the output** (`max_tokens`, word limit) | 🗳️ **Vote on hard cases**, send disagreements to a human |

In [ ]:
# 🏷️ Always record the exact configuration used
CONFIGURATION = dict(provider=ACTIVE_PROVIDER, model=ACTIVE_MODEL, temperature=0.0,
                     json_mode=True, prompt_version="v4", eval_set=EVAL_FINGERPRINT,
                     date=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
display(pd.Series(CONFIGURATION, name="configuration").to_frame())
if ACTIVE_MODEL and not re.search(r"\d", ACTIVE_MODEL):
    callout("The model name contains no version number: check that it is not a moving alias.", "warning")

<div style="background:#C4621D;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">05</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Context, tokens and cost</span><br>
<i style="color:#FBE9DC">What would it cost to code 100,000 survey answers?</i>
</div>

### 5.1 Tokens: the unit of context and of cost

Models read and write **tokens**, not words. OpenAI's rule of thumb for English: **1 token ≈ 4 characters ≈ ¾ of a word**. Many other languages need **more tokens** for the same meaning (Petrov et al., 2023), which weighs directly on the budget of offices working in French, Portuguese, Arabic or African languages.

> **Cost = input tokens × input price + output tokens × output price**
> **Context window ≥ instructions + examples + document + answer**

In [ ]:
sentences = {
    "English": "The consumer price index rose by 2.4 percent in June compared with the previous month.",
    "French": "L'indice des prix à la consommation a augmenté de 2,4 % en juin par rapport au mois précédent.",
    "Portuguese": "O índice de preços ao consumidor subiu 2,4 % em junho em relação ao mês anterior.",
    "Kiswahili": "Fahirisi ya bei za bidhaa kwa mlaji iliongezeka kwa asilimia 2.4 mwezi Juni ikilinganishwa na mwezi uliopita.",
    "Arabic": "ارتفع مؤشر أسعار المستهلك بنسبة 2.4 في المائة في يونيو مقارنة بالشهر السابق.",
}
token_counts = pd.Series({language: count_tokens(s) for language, s in sentences.items()}).sort_values()
ax = token_counts.plot.barh(color=[RAMP[2 + i] for i in range(len(token_counts))], figsize=(8, 3))
ax.bar_label(ax.containers[0], labels=[f"{v} tokens (× {v / token_counts['English']:.2f})" for v in token_counts], padding=3)
ax.set_xlim(0, token_counts.max() * 1.45); ax.set_xticks([])
ax.set_title("The same sentence, a different number of tokens by language")
plt.tight_layout(); plt.show()
print("Method:", TOKEN_METHOD, "· translations provided for illustration")
if "approximation" in TOKEN_METHOD:
    callout("tiktoken is unavailable: a character-based approximation does not reflect real differences between languages. "
            "On Colab or Kaggle, run again with tiktoken to see the gap.", "warning")

### 5.2 Where do the tokens of our prompts go?

Every call sends a **fixed part** (instructions, list, examples) and a **variable part** (the description). Let us see how the versions inflated the fixed part.

In [ ]:
versions = {"v0": messages_v0, "v1": messages_v1, "v2": messages_v2, "v3": messages_v3, "v4": messages_v4, "v5": messages_v5}
sample = EVAL_SET["text"].iloc[0]
sizes = pd.DataFrame({
    v: {"fixed": sum(count_tokens(m["content"]) for m in f(sample) if m["role"] == "system"),
        "variable": sum(count_tokens(m["content"]) for m in f(sample) if m["role"] == "user")}
    for v, f in versions.items()}).T
ax = sizes.plot.bar(stacked=True, color=[DEEP_GREEN, TERRA], figsize=(9, 3.4), rot=0, width=0.6)
for i, total in enumerate(sizes.sum(axis=1)):
    ax.text(i, total + 10, f"{total:.0f}", ha="center", color=INK)
ax.set_ylabel("input tokens per call"); ax.legend(["fixed part (system)", "variable part (case)"], frameon=False)
ax.set_title("Quality has a price: the fixed part dominates")
plt.tight_layout(); plt.show()
display(sizes)

### 5.3 Cost calculator: a worked example

Coding **100,000** descriptions with v4. Prices are **illustrative**: replace them with the real tariff in the configuration cell. Four scenarios:
1. **Baseline**: one call per description;
2. **+ prompt caching**: the fixed part, identical across calls, is billed at a discount by the provider (assumption: 90 %);
3. **+ batching**: 20 descriptions per call, so the fixed part is sent once per batch;
4. **both combined**.

In [ ]:
def cost_scenarios(n_docs, fixed, variable, output, cache_discount=0.90, batch_size=20):
    pin, pout = PRICE_IN_PER_MILLION / 1e6, PRICE_OUT_PER_MILLION / 1e6
    batch_calls = math.ceil(n_docs / batch_size)
    baseline = n_docs * (fixed + variable) * pin + n_docs * output * pout
    cached = n_docs * fixed * pin * (1 - cache_discount) + n_docs * variable * pin + n_docs * output * pout
    batched = (batch_calls * fixed + n_docs * variable) * pin + n_docs * output * pout
    both = (batch_calls * fixed * pin * (1 - cache_discount)) + n_docs * variable * pin + n_docs * output * pout
    return pd.Series({"Baseline": baseline, "+ prompt caching": cached,
                      f"+ batching ({batch_size}/call)": batched, "caching + batching": both})

# Assumptions from the presentation
s = cost_scenarios(100_000, fixed=2_500, variable=50, output=40)
ax = s.plot.barh(color=["#A9B5B0", RAMP[2], RAMP[4], RAMP[6]], figsize=(9, 3.2))
ax.bar_label(ax.containers[0], labels=[f"${v:,.2f}" for v in s], padding=4)
ax.invert_yaxis(); ax.set_xlim(0, s.max() * 1.25); ax.set_xticks([])
ax.set_title("Cost to code 100,000 descriptions (illustrative prices)")
plt.tight_layout(); plt.show()
print(f"Gain from prompt design alone: ÷ {s.iloc[0] / s.iloc[-1]:.0f}")

In [ ]:
# 🎛️ Interactive calculator (if ipywidgets is available) with the real token counts of YOUR v4
fixed_v4, variable_v4 = int(sizes.loc["v4", "fixed"]), int(sizes.loc["v4", "variable"])
try:
    import ipywidgets as w
    def _calc(n_docs, batch_size, cache_discount, output):
        r = cost_scenarios(n_docs, fixed_v4, variable_v4, output, cache_discount / 100, batch_size)
        display(r.map(lambda v: f"${v:,.2f}").to_frame("estimated cost"))
    w.interact(_calc,
               n_docs=w.IntSlider(value=100_000, min=1_000, max=1_000_000, step=1_000, description="documents"),
               batch_size=w.IntSlider(value=20, min=1, max=50, description="batch size"),
               cache_discount=w.IntSlider(value=90, min=0, max=100, step=5, description="discount %"),
               output=w.IntSlider(value=40, min=5, max=500, step=5, description="output tokens"))
except Exception as e:
    print(f"Widgets unavailable ({type(e).__name__}): static version with your v4 token counts.")
    display(cost_scenarios(100_000, fixed_v4, variable_v4, 40).map(lambda v: f"${v:,.2f}").to_frame("estimated cost"))

### 5.4 Batching, for real

We now send the 30 cases in **3 calls of 10 cases** instead of 30 calls, and compare the **tokens actually consumed** and the **accuracy**: a bigger batch costs less, but quality can drop when many items share one call.

In [ ]:
BATCH_FORMAT = """
OUTPUT FORMAT
You receive several descriptions, each preceded by its identifier in square brackets.
Answer with a JSON object only:
{"results": [{"id": "<identifier>", "isco_code": "<four-digit code or UNCODABLE>"}]}"""
SYSTEM_BATCH = SYSTEM_V1 + "\n" + RULES + "\n" + EXAMPLES_TEXT + "\n" + BATCH_FORMAT

def evaluate_in_batches(batch_size=10, label="batches"):
    predicted = {}
    for start in range(0, len(EVAL_SET), batch_size):
        block = EVAL_SET.iloc[start:start + batch_size]
        content = "\n".join(f"[{c.id}] {c.text}" for c in block.itertuples())
        r = LLM.chat([{"role": "system", "content": SYSTEM_BATCH}, {"role": "user", "content": content}],
                     json_mode=True, max_tokens=2048, label=label)
        try:
            for item in json.loads(r["text"])["results"]:
                predicted[str(item["id"])] = str(item["isco_code"]).upper()
        except Exception:
            pass
    return (EVAL_SET["id"].map(predicted) == EVAL_SET["gold_code"]).mean()

batch_accuracy = evaluate_in_batches(10, "batches_10")
b = LLM.summary()
comparison = pd.DataFrame({
    "calls": [len(EVAL_SET), b.loc["batches_10", "calls"]],
    "input tokens": [df_v4["input_tokens"].sum(), b.loc["batches_10", "input_tokens"]],
    "accuracy": [res_v4["strict_accuracy"], batch_accuracy],
}, index=["1 case per call (v4)", "10 cases per call"])
display(comparison.style.format({"accuracy": "{:.0%}", "input tokens": "{:,.0f}", "calls": "{:.0f}"}))
callout("Batching divides the input tokens, but <b>always re-test accuracy</b>: beyond a certain batch size "
        "the model applies the instructions and examples less reliably.", "warning")

### 5.5 Two kinds of caching

| | **Prompt caching** (at the provider) | **Response caching** (on your side) |
|---|---|---|
| Principle | an identical prompt prefix is not recomputed | the answer to an identical request is reused |
| Gain | lower latency and price on the fixed part | free, instant, reproducible call |
| Condition | **stable content first, variable content last** | same model, same prompt, same input, temperature 0 |
| Bonus | — | an **audit trail** for every published figure |

Our client implements **response caching**. Let us rerun exactly the same evaluation:

In [ ]:
LLM.cache.clear()   # start from an empty cache for the demonstration
t0 = time.perf_counter(); _ = evaluate("v4", messages_v4, json_mode=True, show=False, log_it=False); run_1 = time.perf_counter() - t0
t0 = time.perf_counter(); _ = evaluate("v4", messages_v4, json_mode=True, show=False, log_it=False); run_2 = time.perf_counter() - t0
recent = pd.DataFrame(LLM.calls[-60:])
print(f"1st run: {run_1:.2f} s · 2nd run: {run_2:.3f} s"
      + ("  (in simulation the 1st run is already near-instant)" if SIMULATION_MODE else ""))
print(f"Calls served from cache during the 2nd run: {recent.tail(30)['from_cache'].sum()} / 30")
provider_cached = pd.DataFrame(LLM.calls)["provider_cached_tokens"].sum()
print(f"Tokens reported as cached by the provider (where supported): {provider_cached:,}")

<div style="background:#F6E3E0;border-left:5px solid #B83B2E;padding:12px 16px;border-radius:6px">
⛔ <b>A frequent mistake that breaks prompt caching:</b> putting a date, an identifier or the case itself <b>at the beginning</b> of the prompt. A single different character early on invalidates everything after it. Recommended structure: <code>[instructions · list · examples]</code> then <code>[case]</code> — exactly what v1 to v4 do.
</div>

### 5.6 “Lost in the middle”: position matters

Liu et al. (2023) showed that models use information placed at the **beginning or the end** of a long context better than information in the **middle**. Let us test it: a long fictional bulletin contains the March inflation rate, placed at the beginning, in the middle or at the end, together with a **decoy** (the February rate).

In [ ]:
def bulletin(position, value, n_paragraphs=40):
    regions = ["North", "South", "East", "West", "Centre", "Coast", "Plateau", "Savannah"]
    filler = [f"In the {regions[i % 8]} region, economic activity indicator {i} moved by "
              f"{(i * 7) % 11 + 0.5:.1f} points over the period, according to the usual administrative records "
              f"transmitted by the field offices and consolidated by the national accounts division."
              for i in range(n_paragraphs)]
    target = f"At national level, the annual inflation rate for March stands at {value} %."
    decoy = "For reference, the annual inflation rate for February stands at 4.1 %."
    idx = {"beginning": 0, "middle": n_paragraphs // 2, "end": n_paragraphs}[position]
    filler.insert(n_paragraphs // 2 + 5 if position != "middle" else 3, decoy)
    filler.insert(idx if position != "end" else len(filler), target)
    return "MONTHLY BULLETIN OF ECONOMIC CONDITIONS (fictional)\n\n" + "\n".join(filler)

needle_results = []
for position in ["beginning", "middle", "end"]:
    for value in ["7.3", "5.8", "6.2"]:
        doc = bulletin(position, value)
        r = LLM.chat([{"role": "system", "content": "Answer only from the document, with the value and its unit."},
                      {"role": "user", "content": doc + "\n\nQuestion: what is the annual inflation rate for March?"}],
                     max_tokens=512, label="needle")
        needle_results.append(dict(position=position, value=value, correct=value in r["text"],
                                   tokens=count_tokens(doc)))
nr = pd.DataFrame(needle_results)
by_position = nr.groupby("position", sort=False)["correct"].mean()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(3), by_position.values, marker="o", color=TERRA, lw=3, ms=10)
for i, v in enumerate(by_position.values):
    ax.text(i, v + 0.07, f"{v:.0%}", ha="center", color=INK, fontweight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels([f"fact at the {p}" for p in by_position.index])
ax.set_ylim(-0.05, 1.2); ax.set_yticks([]); ax.set_xlim(-0.3, 2.3)
ax.set_title(f"Retrieving a fact by its position (~{nr['tokens'].mean():.0f} tokens of context)")
plt.tight_layout(); plt.show()

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
📌 <b>Less context, better placed.</b> Recent models resist this effect better (you may see that in live mode), but the rule still holds, if only for <b>cost</b>:
<ul>
<li><b>send only what is relevant</b>: the three pages with the table, not the 120-page yearbook (that is what RAG does);</li>
<li><b>summarise or chunk</b> long documents;</li>
<li><b>place key instructions at the edges</b> and restate the essential rule at the end;</li>
<li><b>limit the answer</b> (word limit, <code>max_tokens</code>).</li>
</ul>
</div>

<div style="background:#B83B2E;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">06</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Prompt, RAG or fine-tune?</span><br>
<i style="color:#F8E1DE">Which tool fixes which failure?</i>
</div>

### 6.1 Three tools, three problems

| | ✏️ **Prompting** *(start here)* | 📚 **RAG** *(add knowledge)* | ⚙️ **Fine-tuning** *(change the model)* |
|---|---|---|---|
| **Fixes** | behaviour you can describe in words | missing, changing or citable knowledge | consistent behaviour on a narrow, high-volume task |
| **Data needed** | a few examples | a document corpus | hundreds to thousands of labelled pairs |
| **First result** | hours | days | weeks |
| **Cost** | low | moderate (embeddings, index) | high (training, hosting) |
| **NSO example** | drafting a press release | Q&A on methodology notes | national-scale ISCO coding with a small in-house model |

*Durations and volumes: indicative orders of magnitude.*

### 6.2 Demonstration: when prompting is not enough

A question about a **fictional** survey (Republic of Fictivia): the model cannot know the answer. Without documents, it may **hallucinate** a plausible figure.

In [ ]:
METHODO_NOTE = [
    "P1. The 2025 Labour Force Survey (LFS) of the Republic of Fictivia is conducted by the National Institute of Statistics of Fictivia.",
    "P2. Data collection runs from 3 February to 30 April 2025, through tablet-assisted interviews.",
    "P3. The sample covers 12,480 households across the 14 regions of the country, drawn with a two-stage stratified design.",
    "P4. The overall response rate reaches 91 %, with a minimum of 84 % in the Coast region.",
    "P5. Occupations are coded to ISCO-08 at the four-digit unit group level.",
    "P6. Weights are calibrated on the 2025 population projections by sex, age and area of residence.",
    "P7. Results are representative at national and regional level and by urban or rural area.",
    "P8. Anonymised microdata are available to researchers on request, after signature of a confidentiality agreement.",
]
QUESTION = "What is the sample size of the 2025 LFS of Fictivia?"

without_rag = LLM.chat([{"role": "user", "content": QUESTION}], label="rag")
print("❌ Without documents:", without_rag["text"])

In [ ]:
# 📚 Mini-RAG: TF-IDF retrieval of the relevant passages, then a grounded answer with a citation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# character n-grams: more robust to word variations (household / households…)
vectoriser = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(METHODO_NOTE)
similarities = cosine_similarity(vectoriser.transform([QUESTION]), vectoriser.transform(METHODO_NOTE))[0]
top = similarities.argsort()[::-1][:2]
passages = [METHODO_NOTE[i] for i in top]
display(pd.DataFrame({"passage": [METHODO_NOTE[i][:90] + "…" for i in top], "similarity": similarities[top].round(3)}))

RAG_SYSTEM = ("Answer only from the passages provided and cite the passage used in square brackets, e.g. [P3]. "
              "If the information is not there, answer: “Not found in the documents”.")
with_rag = LLM.chat([{"role": "system", "content": RAG_SYSTEM},
                     {"role": "user", "content": "PASSAGES:\n" + "\n".join(passages) + f"\n\nQUESTION: {QUESTION}"}],
                    label="rag")
print("✅ With RAG:", with_rag["text"])
print("Answer grounded in the source:", "12,480" in with_rag["text"])

all_tokens = count_tokens("\n".join(METHODO_NOTE)); top_tokens = count_tokens("\n".join(passages))
print(f"Context sent: {top_tokens} tokens instead of {all_tokens} for the whole note (÷ {all_tokens / top_tokens:.1f}). "
      "On a 120-page yearbook the gap is far larger.")

### 6.3 A decision path driven by your results

Answer the three questions and the recommended tool appears.

In [ ]:
def recommend(target_reached, factual_failures, consistency_at_scale, confidential_data):
    if target_reached:
        advice, colour = "✅ Deploy, monitor, re-test at every model update.", GREEN
    elif factual_failures:
        advice, colour = "📚 Add retrieval (RAG) and require citations.", TEAL
    elif consistency_at_scale:
        advice, colour = "⚙️ Consider fine-tuning an open-weight model, with hundreds of labelled examples.", BRICK
    else:
        advice, colour = "🔗 Decompose the task into a prompt chain, or collect more labelled examples.", OCHRE
    sovereignty = ("<br>🔒 <b>Confidential data:</b> only a provider under your legal control "
                   "or an open-weight model hosted in house (e.g. via Ollama) is eligible.") if confidential_data else ""
    display(HTML(f'<div style="border-left:6px solid {colour};background:#F4F7F5;padding:12px 16px;border-radius:6px">'
                 f'<b>{advice}</b>{sovereignty}<br><i>Whatever the route: rerun the same evaluation set.</i></div>'))

best = max((j for j in LOG if j["dataset"] == "evaluation" and j["decision"] == "kept"),
           key=lambda j: j["strict_accuracy"])
reached = bool(best["strict_accuracy"] >= TARGET_ACCURACY and best["valid_json"] >= TARGET_JSON)
print(f"Best version kept: {best['version']} ({best['strict_accuracy']:.0%}) · target reached: {reached}")

try:
    import ipywidgets as w
    w.interact(recommend,
               target_reached=w.Checkbox(value=reached, description="Target reached?"),
               factual_failures=w.Checkbox(value=False, description="Failures from missing facts?"),
               consistency_at_scale=w.Checkbox(value=True, description="Consistency at scale needed?"),
               confidential_data=w.Checkbox(value=True, description="Confidential data?"))
except Exception as e:   # ipywidgets missing or incompatible: static version
    print(f"Widgets unavailable ({type(e).__name__}): static display.")
    recommend(reached, factual_failures=False, consistency_at_scale=True, confidential_data=True)

<div style="background:linear-gradient(135deg,#00553A,#00A86A);color:#FFFFFF;padding:16px 22px;border-radius:10px;border-bottom:6px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">07</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">🧪 Your turn (≈ 15 minutes, in pairs)</span><br>
<i style="color:#E6F6EE">Optimise the coding prompt on the frozen set</i>
</div>

| Step | Time | Action |
|---|---|---|
| 1 | 2 min | Read the remaining errors of the best version (cell below) |
| 2 | 3 min | Name the cause of the main errors: ambiguity, context or format? |
| 3 | 8 min | Change **one thing only** in `SYSTEM_V6`, run, log; repeat |
| 4 | 2 min | Check on the holdout set, then record your best version in the shared sheet |

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px">
<b>Rules of the game:</b> never copy an evaluation case into the prompt; one change per version; every version is logged, even when it lowers the score.
</div>

In [ ]:
# 🔍 Step 1: remaining errors of v4
show_errors(df_v4, 12)

In [ ]:
# ✏️ Step 3: your version. Change ONE thing, then run the cell.
RULES_V6 = RULES + """
3. (TO BE COMPLETED: your new rule, or delete this line)"""

SYSTEM_V6 = SYSTEM_V1 + "\n" + RULES_V6 + "\n" + EXAMPLES_TEXT + "\n" + JSON_FORMAT

def messages_v6(text):
    return [{"role": "system", "content": SYSTEM_V6},
            {"role": "user", "content": f"Description: {text}"}]

YOUR_CHANGE = "describe your change here in one sentence"
df_v6, res_v6 = evaluate("v6", messages_v6, json_mode=True, change=YOUR_CHANGE)
show_errors(df_v6, 8)

<details>
<summary><b>💡 Hint (click to reveal)</b></summary>

Look at case **L06** (*Shop assistant in a supermarket*): the model confuses it with market selling. An explicit tie-break rule helps, for example:

`3. Shop sales assistants in a shop or supermarket belong to 5223, not 5211 (which is for markets and stalls).`

Then check on the **holdout set** (case H07) that the gain is real.
</details>

In [ ]:
# ✅ Step 4: control on the holdout set, and decision
_, res_v6_h = evaluate("v6", messages_v6, dataset=HOLDOUT_SET, json_mode=True, change="control")
gain_eval = res_v6["strict_accuracy"] - res_v4["strict_accuracy"]
gain_holdout = res_v6_h["strict_accuracy"] - res_v4_h["strict_accuracy"]
print(f"Gain of v6 over v4 · evaluation set: {gain_eval:+.0%} · holdout set: {gain_holdout:+.0%}")
if gain_eval > 0 and gain_holdout >= 0:
    decide("v6", "kept"); callout("Gain confirmed on both sets: v6 is <b>kept</b>.", "note")
elif gain_eval > 0:
    decide("v6", "to be checked"); callout("Gain on the evaluation set only: risk of overfitting.", "warning")
else:
    decide("v6", "rejected"); callout("No gain: v6 is <b>rejected</b>, but it stays in the log.", "remember")
show_log()

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">08</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Wrap-up, checklist and export</span><br>
<i style="color:#E6F6EE">A good prompt gets an answer. A great prompt gets an answer you can defend.</i>
</div>

### 8.1 Call summary for this session

In [ ]:
summary = LLM.summary()
display(summary.style.format({"cost_usd": "${:.4f}", "mean_latency_s": "{:.2f} s", "input_tokens": "{:,.0f}", "output_tokens": "{:,.0f}"}))
totals = summary[["calls", "from_cache", "input_tokens", "output_tokens", "cost_usd"]].sum()
print(f"Total: {totals['calls']:.0f} calls, {totals['from_cache']:.0f} served from cache · "
      f"{totals['input_tokens']:,.0f} input tokens · {totals['output_tokens']:,.0f} output · "
      f"estimated cost ${totals['cost_usd']:.4f} ({'illustrative' if SIMULATION_MODE else 'configured'} prices)")

### 8.2 The great-prompt checklist

The next cell automatically checks everything that can be checked inside this notebook.

In [ ]:
evaluations = [j for j in LOG if j["dataset"] == "evaluation"]
kept = [j for j in evaluations if j["decision"] == "kept"]
best = max(kept, key=lambda j: j["strict_accuracy"])
checks = [
    ("Evaluation", "Success criteria and target score written down", "TARGET_ACCURACY" in globals()),
    ("Evaluation", "Frozen, versioned set (fingerprint verified)", fingerprint(EVAL_SET) == EVAL_FINGERPRINT),
    ("Quality", "Metric matches the task (strict + per class)", True),
    ("Quality", "Log: one version per change, decisions recorded", all(j["decision"] for j in evaluations)),
    ("Variance", "Temperature set to 0 for production", CONFIGURATION["temperature"] == 0),
    ("Variance", "Agreement across repeated runs measured", "cons_t0" in globals()),
    ("Cost", "Cost per 1,000 documents estimated", "cost_scenarios" in globals()),
    ("Cost", "Stable content first (system), case last", messages_v4("x")[-1]["content"].endswith("x")),
    ("Governance", "No evaluation-set leakage in the version kept",
     not any(f'"{t}"' in globals().get("SYSTEM_" + best["version"].upper(), "") for t in EVAL_SET["text"])),
    ("Governance", "Target reached by the best version kept",
     bool(best["strict_accuracy"] >= TARGET_ACCURACY and best["valid_json"] >= TARGET_JSON)),
]
rows = "".join(
    f'<tr><td style="padding:4px 10px;color:{DEEP_GREEN};font-weight:700">{g}</td><td style="padding:4px 10px">{t}</td>'
    f'<td style="padding:4px 10px;font-size:18px">{"✅" if ok else "⬜"}</td></tr>' for g, t, ok in checks)
display(HTML(f'<table style="border-collapse:collapse">{rows}</table>'))
print(f"{sum(ok for *_, ok in checks)} / {len(checks)} checks passed · best version kept: {best['version']}")

### 8.3 Export for the 14:45 benchmark

The following files are written to the `optimisation_results/` folder:
- `iteration_log.csv`: the full history;
- `best_prompt.json`: the prompt kept and its configuration, to reuse when comparing engines;
- `evaluation_set.csv` and `holdout_set.csv`: the frozen sets, with their fingerprint.

In [ ]:
from pathlib import Path
folder = Path("optimisation_results"); folder.mkdir(exist_ok=True)
pd.DataFrame(LOG).to_csv(folder / "iteration_log.csv", index=False, encoding="utf-8-sig")
EVAL_SET.to_csv(folder / "evaluation_set.csv", index=False, encoding="utf-8-sig")
HOLDOUT_SET.to_csv(folder / "holdout_set.csv", index=False, encoding="utf-8-sig")
best_prompt = globals().get("SYSTEM_" + best["version"].upper(), SYSTEM_V4)
(folder / "best_prompt.json").write_text(json.dumps(dict(
    version=best["version"], system=best_prompt, user_template="Description: {text}",
    configuration=dict(CONFIGURATION, prompt_version=best["version"]),
    scores=dict(strict_accuracy=best["strict_accuracy"], valid_json=best["valid_json"]),
    eval_set_fingerprint=EVAL_FINGERPRINT), ensure_ascii=False, indent=2), encoding="utf-8")
for f in sorted(folder.iterdir()):
    print(f"📄 {f}  ({f.stat().st_size:,} bytes)")
if IN_COLAB:
    print("Colab: open the 📁 panel on the left to download the files.")

### 8.4 Key takeaways

<div style="display:flex;flex-wrap:wrap;gap:10px">
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#00704A">01</div><b>Measure before you change</b><br>A frozen evaluation set turns opinions into numbers. One change per version, everything logged.</div>
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#D49A00">02</div><b>Engineer reproducibility</b><br>Low temperature, pinned model, constrained format, consistency test, verification before publication.</div>
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#C4621D">03</div><b>Design for cost</b><br>Prompt structure, caching and batching can cut the bill tenfold; re-test quality after every optimisation.</div>
</div>

<div style="background:#E8F5EF;border-left:5px solid #00A86A;padding:12px 16px;border-radius:6px;margin-top:12px">
<b>➡️ Next, at 14:45: “Choosing your engine”.</b> You will run <code>best_prompt.json</code> at two providers, including Groq, to compare latency, cost and quality on the same evaluation set.<br>
<b>📤 Share:</b> commit your evaluation set and iteration log to the workshop GitHub organisation, for the STG17 reference manual (activity 4.2.1).<br>
<b>🎯 Commitment:</b> before the end of the week, pick one prompt used in your office, build it a 30-case evaluation set, and record its baseline score.
</div>

### References

- United Nations General Assembly (2014). *Resolution 68/261, Fundamental Principles of Official Statistics.*
- International Labour Organization (2012). *International Standard Classification of Occupations, ISCO-08.*
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* NeurIPS Datasets and Benchmarks.
- Wang, X. et al. (2022). *Self-Consistency Improves Chain of Thought Reasoning in Language Models.* arXiv:2203.11171.
- OpenAI Help Center. *What are tokens and how to count them?*
- Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* NeurIPS 2023.
- Liu, N. F. et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts.* Transactions of the ACL (2024).
- Groq. *Supported Models* (console.groq.com/docs/models), accessed September 2026.

<small><i>All data is fictional. ISCO-08 titles are abridged. In simulation mode the scores come from a teaching simulator and measure no real model. Prices are illustrative.</i></small>